This notebook manually coerces raw Zhang ST AnnData to a standardized format for downstream analysis and concatenation.

Operation(s):

1. Manual curation of metadata
2. Mapping dimensions to reference genes
3. Addition of regional annotations
4. Basic QC filtering

Output(s):
1. DATADIR/7_spatial/harmonized/SP2-02.h5ad

Requirement(s):
1. DATADIR/7_spatial/harmonized/Zhang_spatial.h5ad
2. DATADIR/7_spatial/abc/25um_annotation_mtx.nrrd
3. DATADIR/7_spatial/structure_tree_processed.json
4. DATADIR/7_spatial/gene_metadata.csv

In the next cell you can indicate whether you want to skip the QC part of the notebook, where cells with less than 100 counts in HypoMap3D's HVGs are removed. 

If your goal is to only recreate the atlas, use SKIP_QC = False. 

However, if you want to recreate all HypoMap3D figures from the publication you will need all cells, in which case you use SKIP_QC = True.

In [1]:
SKIP_QC = True

In [2]:
import warnings
import json

from pathlib import Path

import scanpy as sc
import pandas as pd
import numpy as np
import nrrd

from dotenv import dotenv_values
from scipy.sparse import csr_matrix

from hm3d_utils.utils import write_clean_h5ad
from project_utils import FileTracker

warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
# Setting up filepaths from config
CONFIG = dotenv_values("config.env") 
DATADIR = Path(CONFIG['DATADIR'])

OUTDIR = DATADIR / "7_spatial"
(OUTDIR / "harmonized").mkdir(exist_ok=True, parents=True) 

tracker = FileTracker(log_file=DATADIR/'tracked_files.toml')

infile = OUTDIR / "raw" / "Zhang_spatial.h5ad"

if SKIP_QC:
    outfile = OUTDIR / 'harmonized' / 'SP2-02_no_qc.h5ad'
else:
    outfile = OUTDIR / 'harmonized' / 'SP2-02.h5ad'

In [4]:
adata = sc.read_h5ad(infile)
adata

AnnData object with n_obs × n_vars = 709698 × 1122
    obs: 'brain_section_label_GE', 'brain_section_label_meta', 'feature_matrix_label', 'donor_label', 'donor_genotype', 'donor_sex', 'cluster_alias', 'x', 'y', 'z', 'subclass_confidence_score', 'cluster_confidence_score', 'neurotransmitter', 'class', 'subclass', 'supertype', 'cluster', 'neurotransmitter_color', 'class_color', 'subclass_color', 'supertype_color', 'cluster_color', 'x_ccf', 'y_ccf', 'z_ccf', 'parcellation_index', 'parcellation_organ', 'parcellation_category', 'parcellation_division', 'parcellation_structure', 'parcellation_substructure', 'parcellation_organ_color', 'parcellation_category_color', 'parcellation_division_color', 'parcellation_structure_color', 'parcellation_substructure_color'

In [5]:
## Zhang specific fixes

# Convert adata.X to sparse from dense
adata.X = csr_matrix(adata.X)

# Fix gene names that mismatch
gene_symbol_map = {
    'Prrxl1': 'Drgx',
    'Slc9a3r1': 'Nherf1',
    '6430628N08Rik': 'Tmem275',
    '4930452B06Rik': 'Cfap20dc'
}
adata.var = adata.var.rename(index=gene_symbol_map)

### Intialise new AnnData object

In [6]:
# Step 1: load all genes from reference from gene_metadata
gene_df = pd.read_csv(DATADIR / 'gene_metadata.csv', index_col=0)
ref_gene_names = gene_df.index.tolist()

# Step 2: Subset to genes only in reference
valid_genes_mask = np.asarray(adata.var_names.isin(ref_gene_names))
print(f'Genes not in reference: {adata.var_names[~valid_genes_mask].tolist()}')
adata = adata[:, valid_genes_mask].copy()

# Step 2: get index of spatial genes in reference gene list
idx_map = np.array([ref_gene_names.index(g) for g in adata.var_names])

# Step 3: construct reformatted AnnData
adata_harmonized = sc.AnnData(
    X=csr_matrix(
        (
            adata.X.data,
            idx_map[adata.X.indices],
            adata.X.indptr
        ),
        shape=(adata.shape[0], len(ref_gene_names))
    ),
    var=gene_df[['gene_ids']].copy()
)

Genes not in reference: []


In [7]:
adata.obs.keys()

Index(['brain_section_label_GE', 'brain_section_label_meta',
       'feature_matrix_label', 'donor_label', 'donor_genotype', 'donor_sex',
       'cluster_alias', 'x', 'y', 'z', 'subclass_confidence_score',
       'cluster_confidence_score', 'neurotransmitter', 'class', 'subclass',
       'supertype', 'cluster', 'neurotransmitter_color', 'class_color',
       'subclass_color', 'supertype_color', 'cluster_color', 'x_ccf', 'y_ccf',
       'z_ccf', 'parcellation_index', 'parcellation_organ',
       'parcellation_category', 'parcellation_division',
       'parcellation_structure', 'parcellation_substructure',
       'parcellation_organ_color', 'parcellation_category_color',
       'parcellation_division_color', 'parcellation_structure_color',
       'parcellation_substructure_color'],
      dtype='object')

### Adding metadata to .obs

In [8]:
# Manually set dataset ID
adata_harmonized.obs['Dataset'] = 'SP2-02'
adata_harmonized.obs['Dataset'] = adata_harmonized.obs['Dataset'].astype('category')

# Map sample metadata to Sample
adata_harmonized.obs['Sample'] = pd.Categorical(adata.obs['brain_section_label'].values)

# Manually set Author
adata_harmonized.obs['Author'] = 'Zhang'
adata_harmonized.obs['Author'] = adata_harmonized.obs['Author'].astype('category')

# Manually set Technology
adata_harmonized.obs['Technology'] = 'MERFISH'
adata_harmonized.obs['Technology'] = adata_harmonized.obs['Technology'].astype('category')

# Map donor metadata to Donor
adata_harmonized.obs['Donor'] = pd.Categorical(adata.obs['donor_label'].values)

# Map sex metadata to Sex
adata_harmonized.obs['Sex'] = pd.Categorical(adata.obs['donor_sex'].values)

# Set cell ids to dataset_sample_barcode
adata_harmonized.obs.index = (
    adata_harmonized.obs['Dataset'].astype(str) 
    + '_' 
    + adata_harmonized.obs['Sample'].astype(str) 
    + '_' 
    + adata.obs.index.astype(str)
).values

### Adding metadata to .var

In [9]:
# Rename 'gene_ids' to 'gene_stable_ids'
adata_harmonized.var = adata_harmonized.var.rename(columns={'gene_ids': 'gene_stable_ids'})
adata.var.index.name = 'gene_names'

# Add highly variable gene column
hvgs = pd.read_csv(DATADIR / '5_clustering' / 'clean_control_4500.csv', header=None, index_col=0).index.tolist()
adata_harmonized.var['IsHVG'] = adata_harmonized.var_names.isin(hvgs)

# Add a column indicating where a gene was captured by this technology
adata_harmonized.var['IsCaptured'] = adata_harmonized.var_names.isin(adata.var_names)

### Moving coordinates to .obsm

In [10]:
# We copy ccf coordinates as a reformatted dataframe to .obsm
ccf_df = adata.obs[['x_ccf', 'y_ccf', 'z_ccf']].copy()
ccf_df.index = adata_harmonized.obs.index
ccf_df.columns = ['x', 'y', 'z']
adata_harmonized.obsm['coords_ccf_mm'] = ccf_df

In [11]:
# We copy source coordinates as a reformatted dataframe to .obsm
source_df = adata.obs[['x', 'y']].copy()
source_df.index = adata_harmonized.obs.index
adata_harmonized.obsm['coords_source'] = source_df

### Adding regional annotations

In [12]:
# Load annotation voxel grid
annotation_nrrd_fp = OUTDIR / 'abc' / '25um_annotation_mtx.nrrd'
annotation_25um, metadata_25um = nrrd.read(annotation_nrrd_fp)
annotation_25um.shape

(528, 320, 456)

In [13]:
# Convert CCF mm coordinates to voxel coordinates
voxel_df = np.floor(adata_harmonized.obsm['coords_ccf_mm'] * 1000 / 25).astype(int) # mm --> um / 25 um
adata_harmonized.obs['RegionID'] = annotation_25um[
    (
        voxel_df['x'],
        voxel_df['y'],
        voxel_df['z']
    )
]

In [14]:
# Load structure tree
structure_tree_fp = OUTDIR / 'structure_tree_processed.json'
with open(structure_tree_fp, 'r') as fp:
    structure_tree = json.load(fp)

# Add 0 as an out of bounds (OOB) region to tree
structure_tree['0'] = {
    'acronym': 'OOB',
    'name': 'Out of Bound'
}

# Get region acronyms
id_acronym_map = {int(k): v['acronym'] for k, v in structure_tree.items()}
adata_harmonized.obs['RegionAcronym'] = adata_harmonized.obs['RegionID'].map(id_acronym_map).astype('category')

# Get region names
id_name_map = {int(k): v['name'] for k, v in structure_tree.items()}
adata_harmonized.obs['RegionName'] = adata_harmonized.obs['RegionID'].map(id_name_map).astype('category')

### Rough QC filtering

In [15]:
if SKIP_QC == False:

    MIN_HVG_COUNTS = 100

    adata_harmonized.obs['total_counts'] = np.asarray(adata_harmonized.X.sum(axis=1)).ravel()
    adata_harmonized.obs['total_hvg_counts'] = np.asarray(adata_harmonized[:, adata_harmonized.var['IsHVG']].X.sum(axis=1)).ravel()
    adata_harmonized.obs['total_genes'] = np.asarray((adata_harmonized.X > 0).sum(axis=1)).ravel()

    valid_mask = (adata_harmonized.obs['total_hvg_counts'] >= MIN_HVG_COUNTS) 

    print(f"{np.sum(~valid_mask)} cells removed with < {MIN_HVG_COUNTS} highly variable gene counts.")

    adata_harmonized = adata_harmonized[valid_mask]

### Save out new AnnData

In [16]:
tracker_descr = 'Harmonized Zhang MERFISH ST dataset.'

if SKIP_QC:
    tracker_descr += ' Not QC-filtered.'
else:
    tracker_descr += ' Filtered to cells with >= 100 counts in HypoMap3D HVGs.'

with tracker.register(outfile) as tf:
    tf.set_description(tracker_descr)
    write_clean_h5ad(adata_harmonized, tf.filepath)

In [17]:
adata_harmonized

AnnData object with n_obs × n_vars = 709698 × 56939
    obs: 'Dataset', 'Sample', 'Author', 'Technology', 'Donor', 'Sex', 'RegionID', 'RegionAcronym', 'RegionName'
    var: 'gene_stable_ids', 'IsHVG', 'IsCaptured'
    obsm: 'coords_ccf_mm', 'coords_source'